In [1]:
import random

# Fijar la semilla
seed = 42
random.seed(seed)

import gymnasium as gym
import highway_env

from stable_baselines3 import A2C, DQN, PPO
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:

# 1. Crear el entorno estándar de evaluación
env = gym.make("highway-v0", render_mode="rgb_array")

# 2. Diccionario con los modelos del equipo
# Formato: "Nombre del modelo": (Clase del algoritmo, "ruta/al/archivo.zip")
modelos_equipo = {
    "DQN_MO": (DQN, "logs/DQN/best_model.zip"),
    "PPO_AA": (PPO, "logs/PPO/best_model.zip"),
    "A2C_AO": (A2C, "logs/A2C/best_model.zip"),
}

resultados = {}
EVAL_EPISODES = 5  # Número de episodios para probar cada modelo

# 3. Bucle para cargar y evaluar cada modelo automáticamente
for nombre, (Algoritmo, ruta_modelo) in modelos_equipo.items():
    print(f"\nEvaluando a: {nombre}...")
    try:
        # Cargar el modelo del compañero
        agent = Algoritmo.load(ruta_modelo)
        
        # Evaluar la política[cite: 2]
        mean_reward, std_reward = evaluate_policy(
            agent, 
            env, 
            n_eval_episodes=EVAL_EPISODES
        )
        
        resultados[nombre] = (mean_reward, std_reward)
        print(f"-> Resultado {nombre}: {mean_reward:.2f} ± {std_reward:.2f}")
        
    except Exception as e:
        print(f"❌ No se pudo cargar {nombre}. Comprueba la ruta o el algoritmo. Error: {e}")

env.close()

# 4. Mostrar el ranking final ordenado de mayor a menor recompensa media
if resultados:
    print("\n" + "="*30)
    print("🏆 RANKING FINAL DEL EQUIPO 🏆")
    print("="*30)
    ranking = sorted(resultados.items(), key=lambda x: x[1][0], reverse=True)
    for pos, (nombre, (media, std)) in enumerate(ranking, 1):
        print(f"{pos}. {nombre} -> Recompensa: {media:.2f} (± {std:.2f})")
    print("="*30)
    
    mejor_modelo = ranking[0][0]
    print(f"👉 El modelo ganador para entregar es: **{mejor_modelo}**")